# Stokastisk 0.1.0 — Qwen2.5-Coder-7B-Instruct on Kaggle Dual Tesla T4

Run the original, unquantized Hugging Face checkpoint `Qwen/Qwen2.5-Coder-7B-Instruct` using Stokastisk 0.1.0 with its bundled vLLM 0.25.1 runtime.

**Target runtime**
- Python 3.13
- PyTorch 2.11.0+cu130
- CUDA 13.0
- Triton 3.6.0
- 2 × NVIDIA Tesla T4, compute capability SM75
- FP16 inference
- `TRITON_ATTN` attention backend

**Two separate parallelism experiments**
1. Tensor parallelism: TP=2, PP=1.
2. Pipeline parallelism: TP=1, PP=2.

The two experiments use both GPUs, but they are not combined. TP=2 and PP=2 together would require four GPU ranks, not two.

**Notebook conventions:** ordinary Kaggle cells, no `main()` function, no generated Python scripts, no `%%writefile`, and no subprocess wrapper. Run from top to bottom in a fresh Kaggle GPU session.

## 1. Validate Kaggle's Python, CUDA, PyTorch, Triton and GPUs

Enable the **GPU** accelerator in Kaggle notebook settings. This notebook intentionally checks the exact software and hardware target used by the Stokastisk 0.1.0 release tests.

In [1]:
import sys
import torch
import triton

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("Triton:", triton.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

assert sys.version_info[:2] == (3, 13), "This notebook targets Python 3.13."
assert torch.__version__.startswith("2.11.0+cu130"), "Unexpected PyTorch build; do not replace Kaggle CUDA packages blindly."
assert torch.version.cuda == "13.0", "This notebook targets CUDA 13.0."
assert triton.__version__ == "3.6.0", "This notebook targets Triton 3.6.0."
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
assert torch.cuda.device_count() == 2, "This notebook expects exactly two visible GPUs."

for gpu_id in range(2):
    props = torch.cuda.get_device_properties(gpu_id)
    print(
        f"GPU {gpu_id}: {props.name} | "
        f"SM {props.major}.{props.minor} | "
        f"{props.total_memory / 2**30:.2f} GiB"
    )
    assert props.name == "Tesla T4", f"Expected Tesla T4, found {props.name}"
    assert (props.major, props.minor) == (7, 5), "Expected compute capability SM75."

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA runtime: 13.0
Triton: 3.6.0
CUDA available: True
Visible GPU count: 2
GPU 0: Tesla T4 | SM 7.5 | 14.56 GiB
GPU 1: Tesla T4 | SM 7.5 | 14.56 GiB


## 2. Install Stokastisk 0.1.0

Use the same package-index installation pattern as your existing notebook. The Stokastisk wheel integrates the vLLM 0.25.1 runtime and the release-specific components.

If Stokastisk is already installed in the current Kaggle session, this cell may report that the requirement is already satisfied.

In [2]:
!pip install stokastisk --extra-index-url https://ai-biruni.github.io/stokastisk/simple/

Looking in indexes: https://pypi.org/simple, https://ai-biruni.github.io/stokastisk/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 46.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 38.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 69.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 65.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 MB 124.5 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [stokastisk]0 [stokastisk]ntainer-standards]


## 3. Validate Stokastisk, bundled vLLM and the backend policy

Do not upgrade or replace Kaggle's PyTorch, CUDA or Triton packages in this notebook. Confirm the installed SDK and bundled vLLM versions, and check the release's prohibited Flash-family backend policy before loading the model.

In [3]:
import importlib.metadata as metadata
import json
import stokastisk
import torch
import triton

from stokastisk import LLM, StokastiskConfig

print("Stokastisk:", stokastisk.__version__)
print("Bundled vLLM:", metadata.version("vllm"))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Triton:", triton.__version__)

assert stokastisk.__version__ == "0.1.0"
assert metadata.version("vllm") == "0.25.1"
assert torch.cuda.device_count() == 2

policy_report = stokastisk.prohibited_report()
print("\nProhibited backend report:")
print(json.dumps(policy_report, indent=2, default=str))
assert policy_report.get("clean") is True, "A prohibited Flash-family component was detected."

print("\nStokastisk runtime validation passed.")

Stokastisk: 0.1.0
Bundled vLLM: 0.25.1
PyTorch: 2.11.0+cu130
CUDA: 13.0
Triton: 3.6.0

Prohibited backend report:
{
  "distributions": [],
  "modules": [],
  "files": [],
  "native_hits": {},
  "binaries": [],
  "clean": true
}

Stokastisk runtime validation passed.


## 4. Select and inspect the original Hugging Face coding model

**Model:** `Qwen/Qwen2.5-Coder-7B-Instruct`

This is the original Alibaba Qwen instruction-tuned coding checkpoint, not a GPTQ, AWQ, GGUF, GPTQ-Marlin or other quantized derivative. The repository's original weights are BF16; vLLM will cast them to FP16 for T4 inference.

The model uses `Qwen2ForCausalLM`, the architecture already exercised by the supplied Stokastisk notebook. The model's advertised long context is deliberately not enabled here: start with 2,048 tokens to leave a conservative memory budget for the two T4 GPUs.

In [4]:
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
MODEL_ARCHITECTURE = "Qwen2ForCausalLM"

compatibility = stokastisk.check_model(
    MODEL_ID,
    architectures=[MODEL_ARCHITECTURE],
)

print("Model compatibility report:")
print(json.dumps(compatibility, indent=2, default=str))

assert compatibility.get("status") == "SUPPORTED", (
    "Stokastisk did not report this model as supported. "
    "Inspect the compatibility report before proceeding."
)

print("\nSelected model:", MODEL_ID)
print("Expected architecture:", MODEL_ARCHITECTURE)
print("Inference dtype: FP16 (dtype='half')")
print("Checkpoint quantization: none; original HF checkpoint")

Model compatibility report:
{
  "model": "Qwen/Qwen2.5-Coder-7B-Instruct",
  "dtype": "half",
  "quantization": {
    "method": null,
    "status": "SUPPORTED",
    "evidence": "kaggle",
    "note": "no quantization (all Kaggle runs)"
  },
  "architectures": [
    {
      "architecture": "Qwen2ForCausalLM",
      "module": "qwen2",
      "status": "SUPPORTED",
      "evidence": "kaggle",
      "note": "Qwen2.5-0.5B/1.5B-Instruct generated on 2 x T4"
    }
  ],
  "status": "SUPPORTED"
}

Selected model: Qwen/Qwen2.5-Coder-7B-Instruct
Expected architecture: Qwen2ForCausalLM
Inference dtype: FP16 (dtype='half')
Checkpoint quantization: none; original HF checkpoint


## 5. Shared inference prompts for Python 3.13 coding

Both parallelism experiments use the same coding task. The system prompt explicitly targets Python 3.13 and asks for executable, standard-library-first code with type hints and a short explanation.

In [5]:
SYSTEM_PROMPT = (
    "You are an expert Python 3.13 software engineer. "
    "Write correct, executable Python 3.13 code. "
    "Prefer the standard library unless a dependency is necessary. "
    "Use modern type hints, handle edge cases, and explain important decisions."
)

CODING_PROMPT = (
    "Write a Python 3.13 function named summarize_jsonl(path: str) -> dict "
    "that reads a JSON Lines file using UTF-8, skips blank lines, counts valid "
    "JSON objects, counts malformed nonblank lines, and returns a dictionary "
    "with keys 'valid_records' and 'malformed_lines'. "
    "Use pathlib and the standard library. Include a small usage example. "
    "Return the implementation and explanation."
)

print("System prompt and Python 3.13 coding task are ready.")

System prompt and Python 3.13 coding task are ready.


## 6. Tensor parallelism across both T4 GPUs (TP=2, PP=1)

This is the primary two-GPU inference configuration. Each GPU receives a tensor-parallel shard of the model. It follows the successful TP=2 execution pattern from your uploaded notebook.

Configuration:
- `tensor_parallel_size=2`
- `pipeline_parallel_size=1`
- FP16
- `TRITON_ATTN`
- 2,048-token maximum model length
- eager execution
- custom all-reduce disabled for this T4 environment

In [6]:
config_tp2 = StokastiskConfig(
    tensor_parallel_size=2,
    pipeline_parallel_size=1,
    dtype="half",
    attention_backend="TRITON_ATTN",
    max_model_len=2048,
    gpu_memory_utilization=0.88,
    enforce_eager=True,
    disable_custom_all_reduce=True,
)

tp_report = config_tp2.as_dict()
print("TP=2 configuration:")
print(json.dumps(tp_report, indent=2, default=str))

assert tp_report["tensor_parallel_size"] == 2
assert tp_report["pipeline_parallel_size"] == 1
assert tp_report["dtype"] == "half"
assert tp_report["attention_backend"] == "TRITON_ATTN"

with LLM(MODEL_ID, config=config_tp2) as llm:
    active_backend_tp = llm.verify_attention_backend()
    print("\nVerified attention backend:", active_backend_tp)
    assert active_backend_tp == "TRITON_ATTN"

    tp_chat = llm.chat([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": CODING_PROMPT},
    ])

    tp_answer = tp_chat[0].outputs[0].text
    print("\n" + "=" * 72)
    print("QWEN2.5-CODER-7B-INSTRUCT — TENSOR PARALLEL (TP=2)")
    print("=" * 72)
    print(tp_answer)
    assert tp_answer.strip(), "TP=2 returned an empty response."

    print("\nGPU memory during TP=2 inference:")
    for gpu_id in range(2):
        torch.cuda.synchronize(gpu_id)
        free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_id)
        print({
            "gpu": gpu_id,
            "name": torch.cuda.get_device_name(gpu_id),
            "free_GiB": round(free_bytes / 2**30, 2),
            "total_GiB": round(total_bytes / 2**30, 2),
        })

print("\nTP=2 Stokastisk engine closed.")

TP=2 configuration:
{
  "tensor_parallel_size": 2,
  "pipeline_parallel_size": 1,
  "execution_strategy": null,
  "replica_devices": null,
  "dtype": "half",
  "kv_cache_dtype": "auto",
  "attention_backend": "TRITON_ATTN",
  "enforce_eager": true,
  "disable_custom_all_reduce": true,
  "gpu_memory_utilization": 0.88,
  "max_model_len": 2048,
  "structured_output_backend": "xgrammar",
  "apply_nccl_env": true,
  "nccl_env": {
    "NCCL_P2P_DISABLE": "1",
    "NCCL_IB_DISABLE": "1",
    "NCCL_CUMEM_ENABLE": "0"
  }
}
INFO 11-14 13:43:04 [api_utils.py:273] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.88, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'structured_outputs_config': StructuredOutputsConfig(backend='xgrammar', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), 'attention_config':

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

INFO 11-14 13:43:21 [model.py:619] Resolved architecture: Qwen2ForCausalLM
WARNING 11-14 13:43:21 [model.py:2114] Casting torch.bfloat16 to torch.float16.
INFO 11-14 13:43:21 [model.py:1776] Using max model len 2048
INFO 11-14 13:43:21 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 11-14 13:43:21 [vllm.py:1042] Asynchronous scheduling is enabled.
WARNING 11-14 13:43:21 [vllm.py:1096] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 11-14 13:43:21 [vllm.py:1144] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 11-14 13:43:21 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 11-14 13:43:21 [vllm.py:1322] Cudagraph is disabled under eager mode
INFO 11-14 13:

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (9 outputs left out)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(Worker_TP0 pid=184) WARNING 11-14 13:46:22 [jit_monitor.py:129] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=184) WARNING 11-14 13:46:25 [jit_monitor.py:129] Triton kernel JIT compilation during inference: reduce_segments. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.52s/it, est. speed input: 31.44 toks/s, output: 3.54 toks/s]



QWEN2.5-CODER-7B-INSTRUCT — TENSOR PARALLEL (TP=2)
Here's a Python 3.13 function `summarize_jsonl

GPU memory during TP=2 inference:
{'gpu': 0, 'name': 'Tesla T4', 'free_GiB': 1.34, 'total_GiB': 14.56}
{'gpu': 1, 'name': 'Tesla T4', 'free_GiB': 1.34, 'total_GiB': 14.56}
INFO 11-14 13:46:26 [core_client.py:655] [shutdown] MPClient: start timeout=default
INFO 11-14 13:46:26 [core_client.py:657] [shutdown] MPClient: stopping engine manager
(EngineCore pid=169) INFO 11-14 13:46:26 [core.py:1214] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=169) INFO 11-14 13:46:26 [core.py:1333] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=169) INFO 11-14 13:46:26 [core.py:1364] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=169) INFO 11-14 13:46:26 [core.py:1227] [shutdown] EngineCore: exiting busy loop
(EngineCore pid=169) INFO 11-14 13:46:26 [multiproc_executor.py:426] [shutdown] Executor: waiting for worker

## 7. Confirm memory recovery after TP=2

The engine context manager should shut down the inference workers. Check that both GPUs have recovered their available memory before starting the separate pipeline-parallel experiment.

In [7]:
import gc

gc.collect()
for gpu_id in range(2):
    torch.cuda.empty_cache()
    torch.cuda.synchronize(gpu_id)
    free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_id)
    print({
        "gpu": gpu_id,
        "free_GiB_after_TP2": round(free_bytes / 2**30, 2),
        "total_GiB": round(total_bytes / 2**30, 2),
    })

{'gpu': 0, 'free_GiB_after_TP2': 14.46, 'total_GiB': 14.56}
{'gpu': 1, 'free_GiB_after_TP2': 14.46, 'total_GiB': 14.56}


## 8. Pipeline parallelism across both T4 GPUs (TP=1, PP=2)

This is a separate two-GPU experiment. The model's sequential transformer layers are partitioned into two pipeline stages:
- GPU 0: pipeline stage 0
- GPU 1: pipeline stage 1

Configuration:
- `tensor_parallel_size=1`
- `pipeline_parallel_size=2`
- FP16
- `TRITON_ATTN`
- 2,048-token maximum model length
- eager execution
- custom all-reduce disabled

Your uploaded notebook already demonstrated that this PP=2 configuration can initialize and generate with Qwen2ForCausalLM on two T4 GPUs. This cell repeats the test with the 7B coding checkpoint.

In [8]:
config_pp2 = StokastiskConfig(
    tensor_parallel_size=1,
    pipeline_parallel_size=2,
    dtype="half",
    attention_backend="TRITON_ATTN",
    max_model_len=2048,
    gpu_memory_utilization=0.88,
    enforce_eager=True,
    disable_custom_all_reduce=True,
)

pp_report = config_pp2.as_dict()
print("PP=2 configuration:")
print(json.dumps(pp_report, indent=2, default=str))

assert pp_report["tensor_parallel_size"] == 1
assert pp_report["pipeline_parallel_size"] == 2
assert pp_report["dtype"] == "half"
assert pp_report["attention_backend"] == "TRITON_ATTN"

with LLM(MODEL_ID, config=config_pp2) as llm:
    active_backend_pp = llm.verify_attention_backend()
    print("\nVerified attention backend:", active_backend_pp)
    assert active_backend_pp == "TRITON_ATTN"

    pp_chat = llm.chat([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": CODING_PROMPT},
    ])

    pp_answer = pp_chat[0].outputs[0].text
    print("\n" + "=" * 72)
    print("QWEN2.5-CODER-7B-INSTRUCT — PIPELINE PARALLEL (PP=2)")
    print("=" * 72)
    print(pp_answer)
    assert pp_answer.strip(), "PP=2 returned an empty response."

    print("\nGPU memory during PP=2 inference:")
    for gpu_id in range(2):
        torch.cuda.synchronize(gpu_id)
        free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_id)
        print({
            "gpu": gpu_id,
            "name": torch.cuda.get_device_name(gpu_id),
            "free_GiB": round(free_bytes / 2**30, 2),
            "total_GiB": round(total_bytes / 2**30, 2),
        })

print("\nPP=2 Stokastisk engine closed.")

pipeline_parallel is EXPERIMENTAL: Kaggle-verified on 2xT4 (notebooks 08, 13) but slower than TP=1 for a model that fits one T4; use it only for models that need two GPUs


PP=2 configuration:
{
  "tensor_parallel_size": 1,
  "pipeline_parallel_size": 2,
  "execution_strategy": null,
  "replica_devices": null,
  "dtype": "half",
  "kv_cache_dtype": "auto",
  "attention_backend": "TRITON_ATTN",
  "enforce_eager": true,
  "disable_custom_all_reduce": true,
  "gpu_memory_utilization": 0.88,
  "max_model_len": 2048,
  "structured_output_backend": "xgrammar",
  "apply_nccl_env": true,
  "nccl_env": {
    "NCCL_P2P_DISABLE": "1",
    "NCCL_IB_DISABLE": "1",
    "NCCL_CUMEM_ENABLE": "0"
  }
}
INFO 11-14 13:46:56 [api_utils.py:273] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'pipeline_parallel_size': 2, 'gpu_memory_utilization': 0.88, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'structured_outputs_config': StructuredOutputsConfig(backend='xgrammar', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), 'attention_config

(Worker_PP0 pid=802) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(Worker_PP1 pid=803) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(Worker_PP0 pid=802) INFO 11-14 13:47:28 [weight_utils.py:849] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 14.19 GiB. Available RAM: 25.54 GiB.
(Worker_PP0 pid=802) INFO 11-14 13:47:28 [weight_utils.py:872] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:08<00:25,  8.34s/it]


(Worker_PP1 pid=803) INFO 11-14 13:47:38 [model_runner.py:302] Model loading took 7.15 GiB and 11.356968 seconds


... (6 outputs left out)

(Worker_PP0 pid=802) WARNING 11-14 13:48:13 [jit_monitor.py:129] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_PP0 pid=802) WARNING 11-14 13:48:13 [jit_monitor.py:129] Triton kernel JIT compilation during inference: _post_update_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_PP0 pid=802) WARNING 11-14 13:48:15 [jit_monitor.py:129] Triton kernel JIT compilation during inference: reduce_segments. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.04s/it, est. speed input: 35.18 toks/s, output: 3.96 toks/s]


QWEN2.5-CODER-7B-INSTRUCT — PIPELINE PARALLEL (PP=2)
Here's a Python 3.13 function `summarize_jsonl

GPU memory during PP=2 inference:
{'gpu': 0, 'name': 'Tesla T4', 'free_GiB': 2.42, 'total_GiB': 14.56}
{'gpu': 1, 'name': 'Tesla T4', 'free_GiB': 1.93, 'total_GiB': 14.56}
INFO 11-14 13:48:16 [core_client.py:655] [shutdown] MPClient: start timeout=default


(EngineCore pid=787) INFO 11-14 13:48:16 [core.py:1214] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=787) INFO 11-14 13:48:16 [core.py:1333] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=787) INFO 11-14 13:48:16 [core.py:1364] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=787) INFO 11-14 13:48:16 [core.py:1227] [shutdown] EngineCore: exiting busy loop
(EngineCore pid=787) INFO 11-14 13:48:16 [multiproc_executor.py:426] [shutdown] Executor: waiting for worker exit count=2
(Worker_PP0 pid=802) INFO 11-14 13:48:16 [multiproc_executor.py:790] Parent process exited, terminating worker queues
(EngineCore pid=787) INFO 11-14 13:48:21 [multiproc_executor.py:433] [shutdown] Executor: all workers exited gracefully
WARNING 11-14 13:48:21 [utils.py:626] [shutdown] Process manager: force killing remaining processes count=1

PP=2 Stokastisk engine closed.


## 9. Confirm memory recovery after PP=2

In [9]:
gc.collect()
for gpu_id in range(2):
    torch.cuda.empty_cache()
    torch.cuda.synchronize(gpu_id)
    free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_id)
    print({
        "gpu": gpu_id,
        "free_GiB_after_PP2": round(free_bytes / 2**30, 2),
        "total_GiB": round(total_bytes / 2**30, 2),
    })

{'gpu': 0, 'free_GiB_after_PP2': 14.46, 'total_GiB': 14.56}
{'gpu': 1, 'free_GiB_after_PP2': 14.46, 'total_GiB': 14.56}


## 10. Final execution summary

A successful run requires:
1. Python 3.13, Stokastisk 0.1.0, vLLM 0.25.1, PyTorch 2.11.0+cu130 and Triton 3.6.0.
2. Two Tesla T4 GPUs with SM75.
3. `stokastisk.check_model()` reports the Qwen2 architecture as supported.
4. The active attention backend is `TRITON_ATTN` in both runs.
5. TP=2 completes Python coding inference.
6. PP=2 completes Python coding inference.
7. Both engines close and GPU memory is recovered.

**Interpretation:** this notebook tests two ways to distribute the same unquantized 7B model over two T4 GPUs. It is not a throughput benchmark. On PCIe-connected T4s, TP and PP communication overhead can be significant. PP=2 is not necessarily faster than TP=2.

In [10]:
print("=" * 72)
print("STOKASTISK 0.1.0 — QWEN2.5 CODER 7B DUAL-T4 TEST")
print("=" * 72)
print("Model:", MODEL_ID)
print("Architecture:", MODEL_ARCHITECTURE)
print("Precision: FP16")
print("Attention backend: TRITON_ATTN")
print("Tensor-parallel experiment: TP=2, PP=1")
print("Pipeline-parallel experiment: TP=1, PP=2")
print("GPUs:", torch.cuda.device_count())
print("Hardware:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("\nNotebook cells reached the final summary.")
print("Review the TP=2 and PP=2 inference outputs above for successful generation.")

STOKASTISK 0.1.0 — QWEN2.5 CODER 7B DUAL-T4 TEST
Model: Qwen/Qwen2.5-Coder-7B-Instruct
Architecture: Qwen2ForCausalLM
Precision: FP16
Attention backend: TRITON_ATTN
Tensor-parallel experiment: TP=2, PP=1
Pipeline-parallel experiment: TP=1, PP=2
GPUs: 2
Hardware: ['Tesla T4', 'Tesla T4']

Notebook cells reached the final summary.
Review the TP=2 and PP=2 inference outputs above for successful generation.
